In [2]:
# If any import below fails, uncomment the line below, run it, then Restart Runtime:
# !pip install -q scikit-learn tensorflow opencv-python-headless

# ============================================================
# 🔧 Environment & Setup Check
# Run this first. It confirms every library works before we
# start building, and tells you if a GPU is available.
# ============================================================

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

import sklearn
print("scikit-learn version:", sklearn.__version__)

import tensorflow as tf
print("TensorFlow version:", tf.__version__)

import cv2
print("OpenCV version:", cv2.__version__)

gpu_devices = tf.config.list_physical_devices("GPU")
if gpu_devices:
    print(f"\n✅ GPU available: {gpu_devices[0].name} — Project 2 will run faster.")
else:
    print("\nℹ️ No GPU detected — everything will still run, just a bit slower on Project 2.")
    print("   In Colab: Runtime → Change runtime type → Hardware accelerator → GPU")

print("\nAll core libraries imported successfully. You're ready to go! 🚀")

Python version: 3.11.6
NumPy version: 2.3.1
Pandas version: 3.0.3
scikit-learn version: 1.9.0
TensorFlow version: 2.21.0
OpenCV version: 5.0.0

ℹ️ No GPU detected — everything will still run, just a bit slower on Project 2.
   In Colab: Runtime → Change runtime type → Hardware accelerator → GPU

All core libraries imported successfully. You're ready to go! 🚀


In [3]:
movies = pd.DataFrame({
    "title": [
        "Interstellar", "Inception", "The Martian", "Arrival",
        "The Matrix", "Avatar", "Titanic", "The Notebook",
        "Avengers: Endgame", "Iron Man", "Jurassic Park", "The Dark Knight"
    ],
    "description": [
        "space science fiction astronauts future adventure",
        "science fiction dreams technology thriller mind bending",
        "space science fiction astronaut survival mars adventure",
        "science fiction aliens language space mystery",
        "science fiction technology artificial intelligence action",
        "science fiction space aliens adventure fantasy",
        "romance drama ship ocean historical tragedy",
        "romance relationship love drama emotional",
        "superhero action marvel time travel adventure",
        "superhero action technology marvel engineering",
        "dinosaurs science adventure action island",
        "superhero action crime batman thriller"
    ]
})

movies

,title,description
0,Interstellar,space science fiction astronauts future adventure
1,Inception,science fiction dreams technology thriller min...
2,The Martian,space science fiction astronaut survival mars ...
3,Arrival,science fiction aliens language space mystery
4,The Matrix,science fiction technology artificial intellig...
5,Avatar,science fiction space aliens adventure fantasy
6,Titanic,romance drama ship ocean historical tragedy
7,The Notebook,romance relationship love drama emotional
8,Avengers: Endgame,superhero action marvel time travel adventure
9,Iron Man,superhero action technology marvel engineering


In [12]:
# %%
print("Dataset Shape:", movies.shape)

print("\nColumn Names:")
print(movies.columns.tolist())

print("\nMissing Values:")
print(movies.isnull().sum())

display(movies.head())

Dataset Shape: (12, 2)

Column Names:
['title', 'description']

Missing Values:
title          0
description    0
dtype: int64


,title,description
0,Interstellar,space science fiction astronauts future adventure
1,Inception,science fiction dreams technology thriller min...
2,The Martian,space science fiction astronaut survival mars ...
3,Arrival,science fiction aliens language space mystery
4,The Matrix,science fiction technology artificial intellig...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

movie_vectorizer = TfidfVectorizer(stop_words="english")

movie_matrix = movie_vectorizer.fit_transform(
    movies["description"]
)

print("Movie matrix shape:", movie_matrix.shape)

Movie matrix shape: (12, 39)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(movie_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (12, 12)


In [6]:
def recommend_movies(movie_title, number_of_recommendations=5):
    if movie_title not in movies["title"].values:
        return f"Movie '{movie_title}' was not found."

    movie_index = movies.index[
        movies["title"] == movie_title
    ][0]

    similarity_scores = list(
        enumerate(similarity_matrix[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = [
        item for item in similarity_scores
        if item[0] != movie_index
    ]

    recommendations = []

    for index, score in similarity_scores[:number_of_recommendations]:
        recommendations.append({
            "movie": movies.iloc[index]["title"],
            "similarity": round(score, 3)
        })

    return pd.DataFrame(recommendations)

In [7]:
recommend_movies("Interstellar")

,movie,similarity
0,Avatar,0.434
1,The Martian,0.367
2,Arrival,0.291
3,Jurassic Park,0.201
4,The Matrix,0.168


In [8]:
recommend_movies("Iron Man")

,movie,similarity
0,Avengers: Endgame,0.464
1,The Matrix,0.281
2,The Dark Knight,0.275
3,Inception,0.144
4,Jurassic Park,0.121


In [9]:
recommend_movies("Titanic")

,movie,similarity
0,The Notebook,0.298
1,Interstellar,0.000
2,Inception,0.000
3,The Martian,0.000
4,Arrival,0.000


In [13]:
# %%
recommend_movies("Avatar 2")

"Movie 'Avatar 2' was not found."

In [10]:
# %%
import pickle

with open("movie_data.pkl", "wb") as file:
    pickle.dump(movies, file)

with open("similarity_matrix.pkl", "wb") as file:
    pickle.dump(similarity_matrix, file)

print("Movie data and similarity matrix saved successfully!")

Movie data and similarity matrix saved successfully!


In [11]:
# %%
print("Available Movies:\n")

for movie in movies["title"]:
    print("-", movie)

Available Movies:

- Interstellar
- Inception
- The Martian
- Arrival
- The Matrix
- Avatar
- Titanic
- The Notebook
- Avengers: Endgame
- Iron Man
- Jurassic Park
- The Dark Knight
